# Self-Healing Feeder: Two-Tier Edge HIF-Sensor -> Graph FLISR on Branch-10 Realistic Faults & Noise

Runs the pre-registered Branch-10 realistic two-tier fusion study (`research/hif-fusion-realistic/PLAN.md`) on Kaggle GPU
(offline Linux OpenDSS wheels + PyTorch CUDA GPU + frozen `hif_sensor/results/model_int8.npz`, verified SHA-256 `3ba5bd23685d7964a6cea0c94064c1e65a8bf3129d83ac129215d110218397c1`):
1. Loads and verifies the frozen Tier-1 Edge HIF-Sensor (`417` params / `504` bytes int8 Tiny MLP, 4th-order Butterworth
   anti-alias front end at `fc = 2,047.9 Hz` Python / `2,045.1 Hz` LTspice, 12-bit `15.36 kHz` ADC,
   `16` window features over `K = 8` cycles, `PERSIST = 3`).
2. Runs realistic Branch-10 OpenDSS sweeps (intermittent Emanuel arcing `r_eff`, hierarchical zone loading + pre/post drift,
   systematic CT/PT bias + additive `0.5 A` / `0.001 pu` noise floors, physically separated fault vs load-drift residual current,
   synchronized `arc_duty` restrike dropout) + Tier-1 synthesis across **5 arc surfaces** and **7 background disturbances**
   on IEEE 123-bus and unseen IEEE 34-bus (passive + active reconfigurable/looped 0-50% DER grids).
3. Evaluates paired ablations (`60Hz_only` vs `60Hz_plus_edge`) on `ml_full` (150 vs 170 feat — direct 1-to-1 comparison
   against the 31.3% SLG 400 ohm baseline in `results/metrics.json`), `PerNodeGBDT`, `ZoneTreeGNN`,
   `ActivePerNodeGBDT`, and bidirectional edge-aware `ActiveGridGATv2` (+ `diff_p_hif` edge feature).
4. Evaluates Split Conformal Prediction (`aps_rand` primary at 90%/95% with `U_SEED = 12345`, plus `lac`),
   scores pre-registered criteria `R1-R4`, and writes isolated `results/hif_fusion_realistic/*.json` and
   `results/provenance/hif_fusion_realistic.json` without modifying legacy `results/hif_fusion/`.
5. Packages isolated outputs into `/kaggle/working/hif_fusion_realistic_results.zip`.

Simulation only. Nothing here is field data.

In [ ]:
import hashlib, os, shutil, zipfile
from pathlib import Path

EXPECTED_SHA256 = "3ba5bd23685d7964a6cea0c94064c1e65a8bf3129d83ac129215d110218397c1"
DATASET_DIR = "/kaggle/input"
WORK = "/kaggle/working/self-healing-feeder"
if os.path.exists(WORK):
    shutil.rmtree(WORK)

src = next((root for root, _dirs, files in os.walk(DATASET_DIR) if "run.py" in files), None)
if src is None:
    zips = [os.path.join(r, f) for r, _d, fs in os.walk(DATASET_DIR) for f in fs if f.endswith(".zip")]
    for z in zips:
        zipfile.ZipFile(z).extractall("/kaggle/working/_bundle")
    src = next((root for root, _dirs, files in os.walk("/kaggle/working/_bundle") if "run.py" in files), None)
if src is None:
    raise SystemExit("Could not find project bundle in /kaggle/input")
shutil.copytree(src, WORK)
shutil.rmtree("/kaggle/working/_bundle", ignore_errors=True)
os.chdir(WORK)
print("Working dir:", WORK)
print("Top-level:", sorted(os.listdir(WORK)))

# Locate & verify frozen Tier-1 Edge HIF-Sensor model_int8.npz (bundled in hif_sensor/results/ or /kaggle/input)
bundled = Path(WORK) / "hif_sensor" / "results" / "model_int8.npz"
if bundled.is_file():
    MODEL_NPZ = str(bundled)
else:
    model_hits = sorted(Path(DATASET_DIR).rglob("model_int8.npz"))
    MODEL_NPZ = str(model_hits[0]) if model_hits else None
if not MODEL_NPZ:
    raise SystemExit("Frozen model_int8.npz not found in hif_sensor/results/ or /kaggle/input")
sha = hashlib.sha256(Path(MODEL_NPZ).read_bytes()).hexdigest()
print("Found frozen model_int8.npz:", MODEL_NPZ, "| sha256:", sha)
assert sha == EXPECTED_SHA256, f"SHA-256 mismatch: {sha} != {EXPECTED_SHA256}"

In [ ]:
# Install OpenDSS offline from bundled Linux wheels (no Internet required).
import subprocess, sys
if os.path.isdir(os.path.join(WORK, "wheels")):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index",
                    "--find-links", "wheels", "OpenDSSDirect.py==0.9.4"], check=True)
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "OpenDSSDirect.py==0.9.4"], check=True)

import torch, opendssdirect as dss
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("OpenDSS:", dss.Basic.Version())
print("CPU count:", os.cpu_count())

In [ ]:
# Fast verification tests on Kaggle environment before running the full experiment.
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_hif_fusion.py"], check=True)

In [ ]:
# Run the full Branch-10 realistic Two-Tier Edge HIF-Sensor -> Graph FLISR study (quick_mode=False).
n_workers = max(1, os.cpu_count() or 4)
cmd = [sys.executable, "-u", "scripts/hif_fusion_realistic_run.py",
       "--model-npz", MODEL_NPZ, "--workers", str(n_workers)]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# Package isolated results/hif_fusion_realistic, provenance/hif_fusion_realistic.json, and realistic .npz files.
import json
from pathlib import Path

OUT = Path("/kaggle/working/hif_fusion_realistic_results")
if OUT.exists():
    shutil.rmtree(OUT)
(OUT / "hif_fusion_realistic").mkdir(parents=True)
(OUT / "provenance").mkdir(parents=True)
(OUT / "generated").mkdir(parents=True)

for p in sorted((Path(WORK) / "results" / "hif_fusion_realistic").glob("*.json")):
    shutil.copy2(p, OUT / "hif_fusion_realistic" / p.name)
for p in sorted((Path(WORK) / "results" / "provenance").glob("hif_fusion_realistic*.json")):
    shutil.copy2(p, OUT / "provenance" / p.name)
for p in sorted((Path(WORK) / "data" / "generated").glob("hif_fusion_realistic_*.npz")):
    shutil.copy2(p, OUT / "generated" / p.name)

summary = json.loads((OUT / "hif_fusion_realistic" / "summary.json").read_text(encoding="utf-8"))
os.chdir("/kaggle/working")
shutil.rmtree(WORK)
shutil.make_archive("/kaggle/working/hif_fusion_realistic_results", "zip", OUT)

print("=== Pre-registered R1-R4 verdict summary ===")
print(json.dumps(summary["verdicts"], indent=2))
print("=== Headline numbers ===")
print(json.dumps(summary["headline"], indent=2))
print("Saved /kaggle/working/hif_fusion_realistic_results.zip")